In [1]:
import sys
import os
import glob
import pandas as pd
import numpy as np
import SimpleITK as sitk
from pathlib import Path

sys.path.append('/Volumes/T7/Software')
from cardpy.Sample_Data.Data_Paths import *
from cardpy.Data_Import import *
from cardpy.Data_Sorting import *
from cardpy.Data_Processing.DTI import *
from cardpy.Data_Processing.cDTI import *
from cardpy.Colormaps import *
from sklearn.linear_model import LinearRegression
from scipy.stats import gaussian_kde

cDTI_cmaps = cDTI_Colormaps_Generator()


In [2]:
# Configuration: Easy to switch between GT and predicted segmentations
USE_GROUND_TRUTH = True  # Set to False to use predicted segmentations

# Paths (same structure as NewFALVonly.py)
if USE_GROUND_TRUTH:
    # Ground truth segmentation folder
    segmentation_base_path = '/Users/saschastocker/Documents/Stanford/DanEnnis20242025/WholeHeartCropISMRM/cDTIAutomaticSegmentJournalPaper/SmartHealthDataHannumAnnotated'
else:
    # TODO: Update this path when you have predicted segmentations
    segmentation_base_path = '/path/to/predicted/segmentations'

# Raw NIfTI data folder (with bvals/bvecs)
raw_data_base_path = '/Users/saschastocker/Documents/Stanford/DanEnnis20242025/WholeHeartCropISMRM/cDTIAutomaticSegmentJournalPaper/SmartHealthRawData'

root_folders = ['Hannum']  # Same as NewFALVonly.py

# Output CSV file
output_csv_path = '/Users/saschastocker/Documents/Stanford/DanEnnis20242025/WholeHeartCropISMRM/nnUNet/hap_results.csv'

# cDTI processing parameters
num_interp_points = 200
smoothness_level = 'Native'
Helix_Angle_Filter_Settings = dict()
Helix_Angle_Filter_Settings['Linear Filter: Outlier StDev'] = 1
Helix_Angle_Filter_Settings['Spatial Filter: Wall Depth Factor'] = 0.25
Helix_Angle_Filter_Settings['Spatial Filter: Kernel Size'] = 5


In [3]:
def process_slice_for_hap(
    nifti_folder_path,
    segmentation_mask_path,
    slice_idx,
    num_interp_points,
    smoothness_level,
    Helix_Angle_Filter_Settings
):
    """
    Process a single slice to calculate helix angle and pitch.
    
    Parameters:
    -----------
    nifti_folder_path : str
        Path to folder containing .nii, .bvals, .bvecs, .header files
    segmentation_mask_path : str
        Path to the segmentation mask file (.nii)
    slice_idx : int
        Which slice to extract from the 4D NIfTI (0-indexed)
    num_interp_points : int
        Number of interpolation points for cDTI
    smoothness_level : str
        Smoothness level ('Native', 'Low', 'Medium', etc.)
    Helix_Angle_Filter_Settings : dict
        Settings for helix angle filtering
    
    Returns:
    --------
    dict with keys: 'HAP', 'HAR', 'Cardiac_DTI_Metrics', or None if processing fails
    """
    try:
        # Load NIfTI files
        nifti_files = glob.glob(os.path.join(nifti_folder_path, '*.nii'))
        header_files = glob.glob(os.path.join(nifti_folder_path, '*.header'))
        bvals_files = glob.glob(os.path.join(nifti_folder_path, '*.bvals'))
        bvecs_files = glob.glob(os.path.join(nifti_folder_path, '*.bvecs'))
        
        if not (nifti_files and header_files and bvals_files and bvecs_files):
            print(f"  Missing NIfTI files in {nifti_folder_path}")
            return None
        
        NifTi_path = nifti_files[0]
        header_path = header_files[0]
        b_values_path = bvals_files[0]
        b_vectors_path = bvecs_files[0]
        
        # Load and process NIfTI data
        [matrix_stacked, b_vals_stacked, b_vecs_stacked, Header, _, _] = NifTi_Reader(
            NifTi_path, b_values_path, b_vectors_path, header_path
        )
        
        # Extract the specific slice
        if matrix_stacked.ndim == 4:
            matrix_stacked = matrix_stacked[:, :, slice_idx, :]
            matrix_stacked = matrix_stacked[:, :, np.newaxis, :]
        else:
            print(f"  Unexpected matrix shape: {matrix_stacked.shape}")
            return None
        
        # Sort data
        [matrix_sorted, b_vals_sorted, b_vecs_sorted] = stacked2sorted(
            matrix_stacked, b_vals_stacked, b_vecs_stacked
        )
        
        # DTI reconstruction
        [_, _, Eigenvectors, Standard_DTI_Metrics] = DTI_recon(
            matrix_sorted, b_vals_sorted, b_vecs_sorted, tensor_fit='NLLS'
        )
        
        # Load segmentation mask (matching hap.ipynb exactly)
        if not os.path.exists(segmentation_mask_path):
            print(f"  Segmentation mask not found: {segmentation_mask_path}")
            return None
        
        # Match hap.ipynb: extract slice from SimpleITK image BEFORE GetArrayFromImage
        myocardium_mask = sitk.ReadImage(segmentation_mask_path)
        # Extract slice [:, :, 0] from SimpleITK image, then convert to array
        myocardial_mask = sitk.GetArrayFromImage(myocardium_mask[:, :, 0])
        # Match hap.ipynb exactly: nan_to_num then add newaxis
        myocardial_mask = np.nan_to_num(myocardial_mask)[:, :, np.newaxis]
        
        # cDTI reconstruction (matching hap.ipynb exactly)
        [Cardiac_DTI_Metrics, Epi, Endo, Mask] = cDTI_recon(
            myocardial_mask,
            Eigenvectors,
            num_interp_points,
            smoothness_level,
            Helix_Angle_Filter_Settings
        )
        
        # Match hap.ipynb exactly
        myocardial_mask_smoothed = np.copy(Mask)
        myocardial_mask_smoothed_nan = np.copy(myocardial_mask_smoothed)
        myocardial_mask_smoothed_nan = myocardial_mask_smoothed_nan.astype('float')
        myocardial_mask_smoothed_nan[myocardial_mask_smoothed_nan == 0] = np.nan
        
        grid = Endo2Epi_Grid(myocardial_mask_smoothed)
        grid_nan = np.copy(grid)
        grid_nan = grid_nan * myocardial_mask_smoothed_nan
        grid_nan = np.clip(grid_nan, 0.0, 1)
        
        # Calculate HAP and HAR for each slice in the mask (matching hap.ipynb approach)
        HAP_list = []
        HAR_list = []
        
        for slc in range(myocardial_mask_smoothed_nan.shape[2]):
            # Use the same boolean mask from grid_nan to filter both arrays (matching hap.ipynb)
            nan_mask = ~(np.isnan(grid_nan[:, :, slc].flatten()))
            
            E2E_data = grid_nan[:, :, slc].flatten()
            E2E_data = E2E_data[nan_mask]
            
            HA_data = Cardiac_DTI_Metrics['HASF'][:, :, slc].flatten()  # BUGFIX: use spatially-filtered HA (HASF), not raw HA (raw has polarity-flipped border voxels -> shallow pitch)
            HA_data = HA_data[nan_mask]
            
            # Skip if no valid data
            if len(E2E_data) == 0 or len(HA_data) == 0:
                continue
            
            # Linear regression for pitch (matching hap.ipynb exactly)
            try:
                model = LinearRegression().fit(E2E_data[:, np.newaxis], HA_data[:, np.newaxis])
                # Enforcing true 0 and true 1, true epi and endo endpoint/limit
                x_predicted = np.linspace(0, 1, 101)
                y_predicted = model.intercept_ + model.coef_ * x_predicted[:, np.newaxis]
                
                # Adds the helix angle pitches from each slice
                HAP_list.append(float(model.coef_[0]))
                # Get the ranges, positive - (negative) -> range of the angles
                HAR_list.append(float(y_predicted[0] - y_predicted[-1]))
            except Exception as e:
                print(f"    Warning: Regression failed for slice {slc}: {e}")
                continue
        
        # Return average HAP and HAR if multiple slices, or single values
        result = {
            'HAP': np.mean(HAP_list) if HAP_list else np.nan,
            'HAR': np.mean(HAR_list) if HAR_list else np.nan,
            'Cardiac_DTI_Metrics': Cardiac_DTI_Metrics,
            'HAP_per_slice': HAP_list,
            'HAR_per_slice': HAR_list
        }
        
        return result
        
    except Exception as e:
        print(f"  Error processing slice: {e}")
        import traceback
        traceback.print_exc()
        return None


In [ ]:
# Main processing loop
results_list = []

for root_folder in root_folders:
    print(f'Processing root folder: {root_folder}')
    segmentation_root_path = os.path.join(segmentation_base_path, root_folder)
    # Raw data path goes directly to volunteers (no root_folder level)
    raw_data_root_path = raw_data_base_path
    
    # Loop through volunteer folders
    for volunteer_folder in os.listdir(segmentation_root_path):
        if not volunteer_folder.startswith('Volunteer'):
            continue
        
        print(f'\nProcessing {volunteer_folder}')
        volunteer_seg_path = os.path.join(segmentation_root_path, volunteer_folder)
        volunteer_raw_path = os.path.join(raw_data_root_path, volunteer_folder)
        
        distortion_corrected_seg_folder = os.path.join(volunteer_seg_path, 'Distortion_Corrected')
        distortion_corrected_raw_folder = os.path.join(volunteer_raw_path, 'Distortion_Corrected')
        
        if not os.path.isdir(distortion_corrected_seg_folder):
            print(f"  Skipping {volunteer_folder}: no Distortion_Corrected folder")
            continue
        
        # Loop through DiVO and MDDW folders
        for divo_folder in os.listdir(distortion_corrected_seg_folder):
            if not (divo_folder.startswith('DiVO') or divo_folder.startswith('MDDW')):
                continue
            
            print(f'  Processing {divo_folder}')
            divo_seg_path = os.path.join(distortion_corrected_seg_folder, divo_folder)
            divo_raw_path = os.path.join(distortion_corrected_raw_folder, divo_folder)
            
            # Determine number of slices dynamically (like NewFALVonly.py)
            original_folder = os.path.join(divo_seg_path, '01_Original_Images')
            if not os.path.isdir(original_folder):
                print(f"    Skipping: Missing 01_Original_Images folder")
                continue
            
            nii_files = [
                f for f in os.listdir(original_folder)
                if f.endswith('.nii') or f.endswith('.nii.gz')
            ]
            num_slices = len(nii_files)
            
            if num_slices == 0:
                print(f"    Skipping: No NIfTI files found")
                continue
            
            print(f"    Found {num_slices} slices")
            
            # Path to NIfTI folder (with bvals/bvecs)
            nifti_folder = os.path.join(divo_raw_path, 'NifTis')
            if not os.path.isdir(nifti_folder):
                print(f"    Skipping: Missing NifTis folder in raw data")
                continue
            
            # Path to segmentation masks
            mask_folder = os.path.join(divo_seg_path, '06_Segmentation_Masks_CI')
            if not os.path.isdir(mask_folder):
                print(f"    Skipping: Missing 06_Segmentation_Masks_CI folder")
                continue
            
            # Process each slice
            for i in range(1, num_slices + 1):
                slice_num_str = f'{i:03d}'  # e.g., '001', '002', etc.
                mask_file = os.path.join(mask_folder, f'Cropped_Segmentation_Slice_{slice_num_str}.nii')
                
                if not os.path.exists(mask_file):
                    print(f"    Slice {i}: Mask file not found: {mask_file}")
                    continue
                
                print(f"    Processing slice {i}...")
                
                # Process slice (slice_idx is 0-indexed, so i-1)
                result = process_slice_for_hap(
                    nifti_folder_path=nifti_folder,
                    segmentation_mask_path=mask_file,
                    slice_idx=i-1,  # 0-indexed
                    num_interp_points=num_interp_points,
                    smoothness_level=smoothness_level,
                    Helix_Angle_Filter_Settings=Helix_Angle_Filter_Settings
                )
                
                if result is not None:
                    # Store results
                    row = {
                        'root_folder': root_folder,
                        'volunteer': volunteer_folder,
                        'divo_folder': divo_folder,
                        'slice_number': i,
                        'HAP': result['HAP'],
                        'HAR': result['HAR'],
                        'HAP_per_slice': str(result['HAP_per_slice']),  # Convert list to string for CSV
                        'HAR_per_slice': str(result['HAR_per_slice']),
                        'segmentation_type': 'ground_truth' if USE_GROUND_TRUTH else 'predicted'
                    }
                    
                    # Add summary statistics from Cardiac_DTI_Metrics if needed
                    # (You can expand this to include specific metrics)
                    if 'HA' in result['Cardiac_DTI_Metrics']:
                        ha_data = result['Cardiac_DTI_Metrics']['HA']
                        row['HA_mean'] = np.nanmean(ha_data)
                        row['HA_std'] = np.nanstd(ha_data)
                        row['HA_min'] = np.nanmin(ha_data)
                        row['HA_max'] = np.nanmax(ha_data)
                    
                    results_list.append(row)
                    print(f"      ✓ HAP: {result['HAP']:.4f}, HAR: {result['HAR']:.4f}")
                else:
                    print(f"      ✗ Failed to process slice {i}")

print(f'\n\nProcessed {len(results_list)} slices total')


Processing root folder: Hannum

Processing Volunteer_52
  Processing DiVO_15_04
    Found 6 slices
    Processing slice 1...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/dipy/reconst/dti.py:497: RuntimeWarning: invalid value encountered in divide
  return 3 * np.sqrt(6) * determinant((A_squiggle / A_s_norm))
/Users/saschastocker/Documents/Stanford/DanEnnis20242025/WholeHeartCropISMRM/nnUNet/cardpy/Data_Processing/cDTI.py:250: RuntimeWarning: invalid value encountered in scalar divide
  transverse_angle[x, y] = math.acos(np.dot(vector_2, Circ) / (np.linalg.norm(Circ, ord = 2) * np.linalg.norm(vector_2, ord = 2))) * 180 / np.pi          # Calculate transverse angle


Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium


/var/folders/9r/mlgf66555t98p7ptk_n0g21r0000gn/T/ipykernel_51222/1603473675.py:128: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  HAP_list.append(float(model.coef_[0]))
/var/folders/9r/mlgf66555t98p7ptk_n0g21r0000gn/T/ipykernel_51222/1603473675.py:130: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  HAR_list.append(float(y_predicted[0] - y_predicted[-1]))


      ✓ HAP: -5.5441, HAR: 5.5441
    Processing slice 2...


...

/var/folders/9r/mlgf66555t98p7ptk_n0g21r0000gn/T/ipykernel_51222/1603473675.py:128: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  HAP_list.append(float(model.coef_[0]))
/var/folders/9r/mlgf66555t98p7ptk_n0g21r0000gn/T/ipykernel_51222/1603473675.py:130: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  HAR_list.append(float(y_predicted[0] - y_predicted[-1]))


      ✓ HAP: -9.5902, HAR: 9.5902
    Processing slice 6...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/dipy/reconst/dti.py:497: RuntimeWarning: invalid value encountered in divide
  return 3 * np.sqrt(6) * determinant((A_squiggle / A_s_norm))
/Users/saschastocker/Documents/Stanford/DanEnnis20242025/WholeHeartCropISMRM/nnUNet/cardpy/Data_Processing/cDTI.py:250: RuntimeWarning: invalid value encountered in scalar divide
  transverse_angle[x, y] = math.acos(np.dot(vector_2, Circ) / (np.linalg.norm(Circ, ord = 2) * np.linalg.norm(vector_2, ord = 2))) * 180 / np.pi          # Calculate transverse angle


Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Epicardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
Autoadjust Endocardium
      ✓ HAP: 16.2815, HAR: -16.2815


Processed 306 slices total


/var/folders/9r/mlgf66555t98p7ptk_n0g21r0000gn/T/ipykernel_51222/1603473675.py:128: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  HAP_list.append(float(model.coef_[0]))
/var/folders/9r/mlgf66555t98p7ptk_n0g21r0000gn/T/ipykernel_51222/1603473675.py:130: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  HAR_list.append(float(y_predicted[0] - y_predicted[-1]))


In [5]:
# Save results to CSV
if results_list:
    df = pd.DataFrame(results_list)
    df.to_csv(output_csv_path, index=False)
    print(f'\nResults saved to: {output_csv_path}')
    print(f'\nSummary:')
    print(df.groupby(['volunteer', 'divo_folder']).agg({
        'HAP': ['mean', 'std'],
        'HAR': ['mean', 'std'],
        'slice_number': 'count'
    }))
    print(f'\nFirst few rows:')
    print(df.head())
else:
    print('No results to save!')



Results saved to: /Users/saschastocker/Documents/Stanford/DanEnnis20242025/WholeHeartCropISMRM/nnUNet/hap_results.csv

Summary:
                                HAP                   HAR             \
                               mean        std       mean        std   
volunteer    divo_folder                                               
Volunteer_06 DiVO_15_04    0.846924   6.107540  -0.846924   6.107540   
Volunteer_07 DiVO_15_04    8.554756   9.571742  -8.554756   9.571742   
Volunteer_08 DiVO_15_04   -8.844264   6.450538   8.844264   6.450538   
Volunteer_09 DiVO_15_04    2.347263  11.274011  -2.347263  11.274011   
Volunteer_10 DiVO_15_04    1.802349   8.855445  -1.802349   8.855445   
Volunteer_11 DiVO_15_04   -3.473230   8.070837   3.473230   8.070837   
Volunteer_12 DiVO_15_04    7.043336   5.948758  -7.043336   5.948758   
Volunteer_13 DiVO_15_04   -6.486595  14.553356   6.486595  14.553356   
Volunteer_14 DiVO_15_04    1.348439  11.987274  -1.348439  11.987274   
Volunte